# Notebook 9 – Building a Neural Network with PyTorch


## 1. What is a Neural Network?

A neural network is a machine learning model made from connected layers of neurons. Each neuron receives inputs, performs a weighted calculation, adds a bias, and passes the result through an activation function when one is used.

A simple layer can be represented as:

**y = xWᵀ + b**

where:

- `x` = input values
- `W` = weight matrix
- `b` = bias vector
- `y` = output

### Real-world example

For predicting whether an online retail transaction is a **high-value transaction**, the input could contain:

- Quantity
- Unit price
- Purchase hour
- Purchase month

The neural network learns weights for these features and combines them to make a prediction.

### AI/ML usage

Neural networks are widely used for:

- Image classification
- Text classification
- Recommendation systems
- Fraud detection
- Demand prediction
- Customer behavior prediction
- Speech recognition
- Generative AI


## 2. PyTorch and `nn.Module`

PyTorch is a deep learning framework used to create, train, and deploy neural networks.

`torch.nn` contains ready-to-use building blocks for neural networks.

The most important base class for custom models is:

`nn.Module`

When I create a neural network class, I normally inherit from `nn.Module`.

### Why `nn.Module` is important

It helps PyTorch:

- Organize layers
- Track model parameters
- Identify trainable parameters
- Move models between CPU and GPU
- Save and load model states
- Work with optimizers

### Real-world example

A customer classification model can be created as a class derived from `nn.Module`.

### AI/ML usage

Almost every standard PyTorch neural network model is built using `nn.Module` directly or indirectly.


In [1]:
import torch
import torch.nn as nn

print(torch.__version__)
print(nn.Module)

2.14.0+cpu
<class 'torch.nn.modules.module.Module'>


## 3. Constructor: `__init__`

The constructor is the `__init__` method inside the model class.

This is where I define the layers that the model will contain.

A typical structure is:

```python
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(...)
```

`super().__init__()` initializes the `nn.Module` part of the model.

### Real-world example

If I want:

**4 input features → 8 hidden neurons → 6 hidden neurons → 1 output**

I define those layers inside the constructor.

### AI/ML usage

The constructor is where the architecture of a neural network is specified.


In [2]:
class SimpleNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden1 = nn.Linear(4, 8)
        self.hidden2 = nn.Linear(8, 6)
        self.output = nn.Linear(6, 1)

model = SimpleNetwork()
print(model)

SimpleNetwork(
  (hidden1): Linear(in_features=4, out_features=8, bias=True)
  (hidden2): Linear(in_features=8, out_features=6, bias=True)
  (output): Linear(in_features=6, out_features=1, bias=True)
)


## 4. Layers

A neural network is normally built by stacking layers.

In this notebook:

**Input → Hidden Layer 1 → Activation → Hidden Layer 2 → Activation → Output**

Each layer transforms the information it receives.

### Real-world example

For retail transaction classification:

- Input layer receives transaction features.
- First hidden layer learns simple feature combinations.
- Second hidden layer learns more complex combinations.
- Output layer produces the final prediction.

### AI/ML usage

Different architectures use different combinations of layers depending on the problem. Dense layers are common for tabular data, while convolutional and attention-based layers are common in other types of AI systems.


## 5. `nn.Linear`

`nn.Linear` represents a fully connected or dense layer.

Its basic form is:

`nn.Linear(in_features, out_features)`

For example:

`nn.Linear(4, 8)`

means:

- 4 input features
- 8 output neurons

The layer contains:

- A weight matrix
- A bias vector

For `nn.Linear(4, 8)`:

- Weight shape = `(8, 4)`
- Bias shape = `(8,)`
- Total parameters = `8 × 4 + 8 = 40`

### Real-world example

If a transaction has four input features, the first dense layer can transform those four values into eight learned representations.

### AI/ML usage

`nn.Linear` is widely used in:

- Tabular classification
- Regression
- Multilayer perceptrons
- Classification heads
- Final layers of many deep learning architectures


In [3]:
layer = nn.Linear(4, 8)

print("Weight shape:", layer.weight.shape)
print("Bias shape:", layer.bias.shape)
print("Total parameters:", sum(p.numel() for p in layer.parameters()))

Weight shape: torch.Size([8, 4])
Bias shape: torch.Size([8])
Total parameters: 40


## 6. Forward Method

The `forward()` method defines how data moves through the network.

For the required architecture, I want:

**Input → Hidden Layer → Activation → Hidden Layer → Output**

The forward method connects the layers in that order.

### Why is it important?

The constructor tells PyTorch **what layers exist**.

The forward method tells PyTorch **how those layers are used**.

### Real-world example

For a retail transaction, the four input values travel through the hidden layers and finally produce one output value representing the model's prediction score.

### AI/ML usage

The forward pass is used whenever the model receives data and produces predictions.


In [4]:
class RetailNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden1 = nn.Linear(4, 8)
        self.hidden2 = nn.Linear(8, 6)
        self.output = nn.Linear(6, 1)
        self.activation = nn.ReLU()

    def forward(self, x):
        x = self.activation(self.hidden1(x))
        x = self.activation(self.hidden2(x))
        x = self.output(x)
        return x

model = RetailNetwork()
print(model)

RetailNetwork(
  (hidden1): Linear(in_features=4, out_features=8, bias=True)
  (hidden2): Linear(in_features=8, out_features=6, bias=True)
  (output): Linear(in_features=6, out_features=1, bias=True)
  (activation): ReLU()
)


## 7. Activation Layers

An activation function introduces non-linearity into a neural network.

Without non-linear activations, multiple linear layers would still behave like one linear transformation.

A common activation is **ReLU**:

**ReLU(x) = max(0, x)**

In PyTorch:

`nn.ReLU()`

### Real-world example

A neural network predicting customer behavior may need to learn non-linear relationships between quantity, price, time, and other features. ReLU helps the network represent these relationships.

### AI/ML usage

Common activations include:

- ReLU
- Sigmoid
- Tanh
- GELU
- Softmax

The appropriate activation depends on the architecture and task.


In [5]:
activation = nn.ReLU()

values = torch.tensor([-2.0, -0.5, 0.0, 1.0, 3.0])
result = activation(values)

print("Input:", values)
print("ReLU output:", result)

Input: tensor([-2.0000, -0.5000,  0.0000,  1.0000,  3.0000])
ReLU output: tensor([0., 0., 0., 1., 3.])


## 8. Output Layer

The output layer produces the final value from the neural network.

The number of output neurons depends on the task.

### Common cases

| Task | Typical output |
|---|---|
| Binary classification | 1 neuron |
| Multi-class classification | Number of classes |
| Regression | 1 neuron for one target |

For this notebook, I use **1 output neuron** because I will demonstrate binary classification.

### Important point

For a binary classification model using `BCEWithLogitsLoss`, the output layer normally produces a raw score called a **logit**. A sigmoid can be applied later when I need a probability.

### Real-world example

A retail transaction model can output one value representing whether the transaction belongs to the high-value class.

### AI/ML usage

Output layers are used to convert learned representations into task-specific predictions.


## 9. Model Object

After defining the class, I create an object from that class.

For example:

`model = RetailNetwork()`

This object contains:

- The layers
- Their parameters
- The forward logic
- The complete model architecture

The model object can then be used for:

- Prediction
- Training
- Parameter inspection
- Saving and loading


In [6]:
model = RetailNetwork()

sample_input = torch.randn(3, 4)
output = model(sample_input)

print("Input shape:", sample_input.shape)
print("Output shape:", output.shape)
print("Output:", output)

Input shape: torch.Size([3, 4])
Output shape: torch.Size([3, 1])
Output: tensor([[-0.4017],
        [-0.5338],
        [-0.3700]], grad_fn=<AddmmBackward0>)


## 10. Understanding Input Features

The number of input features must match the number of values provided for each sample.

For this example I use four features:

1. Quantity
2. UnitPrice
3. Hour
4. Month

Therefore:

**Number of input features = 4**

The first layer is:

`nn.Linear(4, 8)`

This means every sample should have four input values.

### Real-world example

A customer transaction can be represented as:

`[quantity, price, hour, month]`

The model receives these four values together.

### AI/ML usage

Choosing useful input features is an important part of building models for structured or tabular data.


## 11. Understanding Hidden Neurons

A hidden layer contains neurons that learn intermediate representations.

Here:

`nn.Linear(4, 8)`

has **8 hidden neurons**.

The next layer:

`nn.Linear(8, 6)`

has **6 hidden neurons**.

So the hidden part of my network is:

**4 → 8 → 6**

There is no universal number of hidden neurons that works for every problem. It depends on the dataset, task, architecture, and amount of data.

### Real-world example

The first hidden layer can learn combinations of transaction features, while the second hidden layer can combine those learned representations into more useful patterns.

### AI/ML usage

Hidden neurons are the main learned representations in a feed-forward neural network.


## 12. Understanding Output Neurons

The output layer is:

`nn.Linear(6, 1)`

So the model has:

**1 output neuron**

This is appropriate for the binary classification example in this notebook.

If I had 3 mutually exclusive classes, I would typically use:

`nn.Linear(6, 3)`

### Key idea

The number of output neurons is determined by the prediction target and the chosen loss/output setup.


## 13. Model Architecture

My complete architecture is:

**Input (4) → Linear (8) → ReLU → Linear (6) → ReLU → Linear (1)**

A simple visual representation is:

```text
Quantity ─┐
UnitPrice ├──→ Hidden Layer 1 → ReLU → Hidden Layer 2 → ReLU → Output
Hour ─────┤          8 neurons          6 neurons         1 neuron
Month ────┘
```

This is a simple fully connected neural network, also called a multilayer perceptron.


## 14. Load the Online Retail Dataset

I am using the provided **Online Retail** dataset for the practical part.

The dataset contains transaction-level information such as:

- Invoice number
- Product code
- Description
- Quantity
- Invoice date
- Unit price
- Customer ID
- Country

For this notebook, I will create a simple binary target called `HighValueOrder`.

The target is based on whether the transaction amount is above the median transaction amount in the cleaned sample.

This is mainly a learning example to connect PyTorch architecture concepts with a real dataset.


In [7]:
import pandas as pd
import numpy as np

df = pd.read_csv("data.csv", encoding="latin1")

df = df.dropna(subset=["Quantity", "UnitPrice", "InvoiceDate"])
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)].copy()

df["TotalAmount"] = df["Quantity"] * df["UnitPrice"]
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

df["Hour"] = df["InvoiceDate"].dt.hour
df["Month"] = df["InvoiceDate"].dt.month

df = df[["Quantity", "UnitPrice", "Hour", "Month", "TotalAmount"]].dropna()

df = df.sample(n=min(20000, len(df)), random_state=42)

threshold = df["TotalAmount"].median()
df["HighValueOrder"] = (df["TotalAmount"] > threshold).astype(int)

df.head()

,Quantity,UnitPrice,Hour,Month,TotalAmount,HighValueOrder
269788,2,7.95,11,7,15.90,1
83138,4,4.25,9,2,17.00,1
538681,3,4.15,14,12,12.45,1
58905,1,0.65,16,1,0.65,0
388858,6,0.85,15,10,5.10,0


## 15. Prepare Features for PyTorch

The model needs numerical input features.

I will use:

- `Quantity`
- `UnitPrice`
- `Hour`
- `Month`

The target is:

- `HighValueOrder`

I standardize the four input features so that their scales are more comparable.

### Why standardize?

For example, price and month can have very different numerical ranges. Standardization can make optimization easier.

The standardized value is approximately:

**z = (x - mean) / standard deviation**



In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

feature_columns = ["Quantity", "UnitPrice", "Hour", "Month"]

X = df[feature_columns].values.astype(np.float32)
y = df["HighValueOrder"].values.astype(np.float32).reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

X_train_tensor = torch.tensor(X_train)
X_test_tensor = torch.tensor(X_test)
y_train_tensor = torch.tensor(y_train)
y_test_tensor = torch.tensor(y_test)

print("Number of input features:", X_train_tensor.shape[1])
print("Training samples:", X_train_tensor.shape[0])
print("Testing samples:", X_test_tensor.shape[0])

Number of input features: 4
Training samples: 16000
Testing samples: 4000


## 16. Build the Final Model

Now I am building the complete model for the dataset.

Architecture:

**4 input features → 8 neurons → ReLU → 6 neurons → ReLU → 1 output neuron**

The output is one logit for binary classification.


In [9]:
class RetailClassifier(nn.Module):
    def __init__(self, input_features, hidden1, hidden2, output_features):
        super().__init__()
        self.layer1 = nn.Linear(input_features, hidden1)
        self.activation1 = nn.ReLU()
        self.layer2 = nn.Linear(hidden1, hidden2)
        self.activation2 = nn.ReLU()
        self.output = nn.Linear(hidden2, output_features)

    def forward(self, x):
        x = self.layer1(x)
        x = self.activation1(x)
        x = self.layer2(x)
        x = self.activation2(x)
        x = self.output(x)
        return x

model = RetailClassifier(
    input_features=4,
    hidden1=8,
    hidden2=6,
    output_features=1
)

print(model)

RetailClassifier(
  (layer1): Linear(in_features=4, out_features=8, bias=True)
  (activation1): ReLU()
  (layer2): Linear(in_features=8, out_features=6, bias=True)
  (activation2): ReLU()
  (output): Linear(in_features=6, out_features=1, bias=True)
)


## 17. Inspect the Model Architecture

Printing the model object gives a useful summary of the layers and their order.

This helps me check whether the architecture I designed is actually the architecture implemented in PyTorch.


In [10]:
print(model)

RetailClassifier(
  (layer1): Linear(in_features=4, out_features=8, bias=True)
  (activation1): ReLU()
  (layer2): Linear(in_features=8, out_features=6, bias=True)
  (activation2): ReLU()
  (output): Linear(in_features=6, out_features=1, bias=True)
)


## 18. Inspect Model Parameters

A neural network learns parameters during training.

The main trainable parameters in a linear layer are:

- Weight matrix
- Bias vector

For a layer with `in_features` and `out_features`:

**Weight parameters = in_features × out_features**

**Bias parameters = out_features**

Therefore:

**Total = in_features × out_features + out_features**


In [11]:
for name, parameter in model.named_parameters():
    print(name)
    print("Shape:", tuple(parameter.shape))
    print("Trainable:", parameter.requires_grad)
    print()

layer1.weight
Shape: (8, 4)
Trainable: True

layer1.bias
Shape: (8,)
Trainable: True

layer2.weight
Shape: (6, 8)
Trainable: True

layer2.bias
Shape: (6,)
Trainable: True

output.weight
Shape: (1, 6)
Trainable: True

output.bias
Shape: (1,)
Trainable: True



## 19. Number of Parameters

For this architecture:

### Layer 1

`4 → 8`

Weights:

`4 × 8 = 32`

Bias:

`8`

Total:

`40`

### Layer 2

`8 → 6`

Weights:

`8 × 6 = 48`

Bias:

`6`

Total:

`54`

### Output layer

`6 → 1`

Weights:

`6 × 1 = 6`

Bias:

`1`

Total:

`7`

### Complete model

**40 + 54 + 7 = 101 trainable parameters**

So this small network has **101 trainable parameters**.


In [12]:
total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameters = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)

print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)

Total parameters: 101
Trainable parameters: 101


## 20. Inspect Weight Matrices

Each `nn.Linear` layer contains a weight matrix.

For the first layer:

`nn.Linear(4, 8)`

the weight matrix has shape:

**8 × 4**

Each of the 8 neurons has one weight for each of the 4 input features.

For the second layer:

**6 × 8**

For the output layer:

**1 × 6**

These values are initially learned from random initialization and later updated during training.


In [14]:
print("Layer 1 weight matrix:")
print(model.layer1.weight)
print("Shape:", model.layer1.weight.shape)

print("Layer 2 weight matrix:")
print(model.layer2.weight)
print("Shape:", model.layer2.weight.shape)

print("Output weight matrix:")
print(model.output.weight)
print("Shape:", model.output.weight.shape)

Layer 1 weight matrix:
Parameter containing:
tensor([[ 0.1580,  0.2176, -0.3268,  0.0651],
        [ 0.0023,  0.4086, -0.0843, -0.4741],
        [ 0.2622,  0.2907, -0.0046, -0.0557],
        [ 0.3419, -0.1937, -0.1275, -0.2078],
        [ 0.0877, -0.0277, -0.0432,  0.3035],
        [ 0.3842, -0.3472, -0.0131,  0.2545],
        [-0.4016, -0.2360, -0.4900, -0.4147],
        [-0.0048,  0.2438, -0.3634,  0.4946]], requires_grad=True)
Shape: torch.Size([8, 4])
Layer 2 weight matrix:
Parameter containing:
tensor([[ 0.3077, -0.2480,  0.3281,  0.2821, -0.0736,  0.1505,  0.0579,  0.0467],
        [ 0.3336, -0.1653,  0.1042, -0.1783,  0.2669, -0.2723,  0.3194,  0.1508],
        [-0.0606, -0.1121, -0.1336,  0.1781, -0.3389,  0.0268,  0.1243, -0.3053],
        [ 0.1187,  0.2875,  0.0323,  0.2962,  0.1225, -0.0372,  0.2093,  0.2161],
        [-0.2256,  0.2493, -0.0583, -0.0516,  0.2712,  0.3215, -0.2557,  0.2638],
        [ 0.1922,  0.3276, -0.0253,  0.0110,  0.0687,  0.2652, -0.2683, -0.1872]],
  

## 21. Inspect Bias Vectors

Every linear layer also has a bias vector by default.

For this model:

- Layer 1 bias shape = `(8,)`
- Layer 2 bias shape = `(6,)`
- Output bias shape = `(1,)`

Bias allows each neuron to shift its activation independently of the weighted input.


In [16]:
print("Layer 1 bias:")
print(model.layer1.bias)

print("Layer 2 bias:")
print(model.layer2.bias)

print("Output bias:")
print(model.output.bias)

Layer 1 bias:
Parameter containing:
tensor([-0.0222,  0.1080, -0.4642, -0.0269, -0.0347,  0.2176,  0.4193,  0.0158],
       requires_grad=True)
Layer 2 bias:
Parameter containing:
tensor([ 0.0374,  0.0286,  0.3340,  0.1467,  0.2486, -0.2294],
       requires_grad=True)
Output bias:
Parameter containing:
tensor([-0.2562], requires_grad=True)


## 22. Trainable vs Non-Trainable Parameters

A parameter with:

`requires_grad = True`

is trainable by default.

During training, PyTorch calculates gradients for trainable parameters and an optimizer updates them.

This is important because not every tensor used in a model must necessarily be trainable.

### AI/ML usage

Freezing parameters is useful when working with pre-trained models and transfer learning.


In [17]:
for name, parameter in model.named_parameters():
    print(name, "->", parameter.requires_grad)

layer1.weight -> True
layer1.bias -> True
layer2.weight -> True
layer2.bias -> True
output.weight -> True
output.bias -> True


## 25. Check the Model on Test Data

After training, I can use the test data to generate predictions.

A sigmoid converts logits into values between 0 and 1.

I use `0.5` as the classification threshold for this simple demonstration.


In [18]:
model.eval()

with torch.no_grad():
    test_logits = model(X_test_tensor)
    test_probabilities = torch.sigmoid(test_logits)
    test_predictions = (test_probabilities >= 0.5).float()

accuracy = (test_predictions == y_test_tensor).float().mean()

print("Test accuracy:", accuracy.item())
print("First five probabilities:")
print(test_probabilities[:5].flatten())

Test accuracy: 0.5419999957084656
First five probabilities:
tensor([0.4939, 0.4843, 0.4972, 0.5183, 0.4923])
